In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Felles steg for alle bilder (samme som i originalnotebooken)
felles = [
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
]

# Uten augmentation
transform_plain = transforms.Compose(felles)

# Med augmentation (brukes kun på treningsdata)
transform_aug = transforms.Compose([
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
] + felles)

train_plain = datasets.MNIST('./data', train=True, download=True, transform=transform_plain)
train_aug = datasets.MNIST('./data', train=True, download=True, transform=transform_aug)
test_ds = datasets.MNIST('./data', train=False, download=True, transform=transform_plain)

# Samme splitt for begge forsøkene
idx = torch.randperm(len(train_plain))
val_idx = idx[:12000]
train_idx = idx[12000:12000 + 4800]   # 10 % av de 48 000 treningsbildene

val_loader = DataLoader(Subset(train_plain, val_idx), batch_size=256)
test_loader = DataLoader(test_ds, batch_size=256)


def evaluate(model, loader, criterion):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss_sum += criterion(out, y).item() * y.size(0)
            correct += (out.argmax(1) == y).sum().item()
            total += y.size(0)
    return loss_sum / total, 100 * correct / total


def run_experiment(train_dataset, num_epochs=30):
    torch.manual_seed(0)  # samme startvekter i begge forsøkene
    model = LeNet5(num_of_classes=10, p=0.5).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0005)
    train_loader = DataLoader(Subset(train_dataset, train_idx), batch_size=64,
                              shuffle=True, drop_last=True)

    hist = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    for epoch in range(num_epochs):
        model.train()
        loss_sum, correct, total = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * y.size(0)
            correct += (out.argmax(1) == y).sum().item()
            total += y.size(0)

        val_loss, val_acc = evaluate(model, val_loader, criterion)
        hist['train_loss'].append(loss_sum / total)
        hist['train_acc'].append(100 * correct / total)
        hist['val_loss'].append(val_loss)
        hist['val_acc'].append(val_acc)
        print(f"Epoke {epoch+1}: train acc {hist['train_acc'][-1]:.2f} %, val acc {val_acc:.2f} %")

    _, test_acc = evaluate(model, test_loader, criterion)
    print(f"Test accuracy: {test_acc:.2f} %")
    return hist


hist_plain = run_experiment(train_plain)
hist_aug = run_experiment(train_aug)

# Plott
epochs = range(1, len(hist_plain['train_loss']) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, hist_plain['train_loss'], 'b--', label='Train (no aug)')
ax1.plot(epochs, hist_plain['val_loss'], 'b-', label='Val (no aug)')
ax1.plot(epochs, hist_aug['train_loss'], 'r--', label='Train (aug)')
ax1.plot(epochs, hist_aug['val_loss'], 'r-', label='Val (aug)')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Loss')
ax1.legend(); ax1.grid()

ax2.plot(epochs, hist_plain['train_acc'], 'b--', label='Train (no aug)')
ax2.plot(epochs, hist_plain['val_acc'], 'b-', label='Val (no aug)')
ax2.plot(epochs, hist_aug['train_acc'], 'r--', label='Train (aug)')
ax2.plot(epochs, hist_aug['val_acc'], 'r-', label='Val (aug)')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)'); ax2.set_title('Accuracy')
ax2.legend(); ax2.grid()

plt.tight_layout()
plt.show()

NameError: name 'LeNet5' is not defined